# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [1]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [2]:
# Add as many cells as you need

# -------------------------------------------------
# Step 1: Load Dataset & Split Features/Target
# -------------------------------------------------
df = pd.read_csv('dataset_milestone1_part3.csv')

X = df.drop(columns=['taxvaluedollarcnt'])
y = df['taxvaluedollarcnt']

# -------------------------------------------------
# Step 2: Train/Test Split
# -------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

In [3]:
# -------------------------------------------------
# Step 3: Feature Scaling (CRITICAL — No Data Leakage)
# -------------------------------------------------
scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply same transformation to test data
X_test_scaled = scaler.transform(X_test)

In [4]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (56632, 17)
Test shape: (14159, 17)


In [5]:
print("List of remaining features after performing Milestone1-Part3:")
print(X.columns.to_list())

List of remaining features after performing Milestone1-Part3:
['airconditioningtypeid', 'bathroomcnt', 'bedroomcnt', 'buildingqualitytypeid', 'calculatedfinishedsquarefeet', 'garagecarcnt', 'garagetotalsqft', 'heatingorsystemtypeid', 'latitude', 'longitude', 'lotsizesquarefeet', 'propertylandusetypeid', 'propertyzoningdesc', 'regionidcounty', 'roomcnt', 'unitcnt', 'yearbuilt']


### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [6]:
# Repeated CV setup with 5 folds and 5 repeats
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)

# Dictionary of Models
models = {
    "Bagging": BaggingRegressor(random_state=random_state),
    "RandomForest": RandomForestRegressor(random_state=random_state),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=random_state)
}

results = {}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train_scaled,
        y_train,
        scoring='neg_mean_absolute_error',
        cv=cv,
        n_jobs=-1
    )
    mae_scores = -scores
    results[name] = {
        "mean_MAE": mae_scores.mean(),
        "std_MAE": mae_scores.std()
    }

results

{'Bagging': {'mean_MAE': np.float64(209263.49732447285),
  'std_MAE': np.float64(3411.156890526315)},
 'RandomForest': {'mean_MAE': np.float64(201044.95386286717),
  'std_MAE': np.float64(3456.0271251049994)},
 'HistGradientBoosting': {'mean_MAE': np.float64(204429.27733393182),
  'std_MAE': np.float64(3477.905651363502)}}

In [7]:
# Better display of results
pd.DataFrame(results)

,Bagging,RandomForest,HistGradientBoosting
mean_MAE,209263.497324,201044.953863,204429.277334
std_MAE,3411.156891,3456.027125,3477.905651


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

**Answer**

| Model                  | Mean MAE | Std MAE |
|-----------------------|----------|---------|
| Bagging               | 209,263  | 3,411   |
| Random Forest         | 201,045  | 3,456   |
| HistGradientBoosting  | 204,429  | 3,478   |



**Best performing model:** 
- The **Random Forest** model performed best overall, achieving the **lowest mean MAE (~201,045)**, indicating it provides the most accurate predictions among the three baseline models. This suggests it is better at capturing the underlying patterns in the data compared to Bagging and HistGradientBoosting with <u>default parameters</u>.


**Most stable model:**
- The **Bagging** model had the **lowest standard deviation (~3,411)**, making it the <u>most stable</u> across different cross-validation folds. However, its higher MAE indicates that while it is consistent, it is not as accurate as the other models.


**Overfitting / Underfitting:**
- There are no strong signs of **overfitting**, as all models show relatively low and similar standard deviations, indicating consistent performance across folds. However, there may be **slight underfitting**, especially for Bagging, since its error is noticeably higher. Additionally, the fact that all models are using **default parameters** suggests that **further tuning could improve performance**, particularly for HistGradientBoosting, which is expected to perform better after optimization.


### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [8]:
# Feature Engineering Summary (From Milestone 1 - Part 5)
#
#     1. Log Transformation Features
#       log_sqft = log(1 + calculatedfinishedsquarefeet)
#       log_lotsize = log(1 + lotsizesquarefeet)
#
#     2. Interaction Features
#       sqft_x_bath = calculatedfinishedsquarefeet * bathroomcnt
#       bed_bath_ratio = bedroomcnt / (bathroomcnt + 1)
#


In [9]:
# -----------------------------------------
# Step 1: Add Engineered Features
# -----------------------------------------

# Copy original training data
X_train_fe = X_train.copy()

# -------------------------
# Log Features
# -------------------------
X_train_fe['log_sqft'] = np.log1p(X_train_fe['calculatedfinishedsquarefeet'])
X_train_fe['log_lotsize'] = np.log1p(X_train_fe['lotsizesquarefeet'])

# -------------------------
# Interaction Features
# -------------------------
X_train_fe['sqft_x_bath'] = X_train_fe['calculatedfinishedsquarefeet'] * X_train_fe['bathroomcnt']
X_train_fe['bed_bath_ratio'] = X_train_fe['bedroomcnt'] / (X_train_fe['bathroomcnt'] + 1)


In [10]:
# -------------------------------------
# Step 2: Scale Features
# -------------------------------------
scaler = StandardScaler()

X_train_fe_scaled = scaler.fit_transform(X_train_fe)

In [11]:
# ---------------------------------------------
# Step 3: Re-run Models with CV
# ---------------------------------------------

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)

models = {
    "Bagging": BaggingRegressor(random_state=random_state),
    "RandomForest": RandomForestRegressor(random_state=random_state),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=random_state)
}

results_fe = {}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train_fe_scaled,
        y_train,
        scoring='neg_mean_absolute_error',
        cv=cv,
        n_jobs=-1
    )
    
    mae_scores = -scores
    
    results_fe[name] = {
        "mean_MAE": mae_scores.mean(),
        "std_MAE": mae_scores.std()
    }

results_fe

{'Bagging': {'mean_MAE': np.float64(210665.55989865644),
  'std_MAE': np.float64(3464.5295266417456)},
 'RandomForest': {'mean_MAE': np.float64(202275.86038910906),
  'std_MAE': np.float64(3298.87090728044)},
 'HistGradientBoosting': {'mean_MAE': np.float64(205014.8322832639),
  'std_MAE': np.float64(3348.417164251742)}}

In [12]:
{'Bagging': {'mean_MAE': np.float64(210665.55989865644),
  'std_MAE': np.float64(3464.5295266417456)},
 'RandomForest': {'mean_MAE': np.float64(202275.86038910906),
  'std_MAE': np.float64(3298.87090728044)},
 'HistGradientBoosting': {'mean_MAE': np.float64(205014.8322832639),
  'std_MAE': np.float64(3348.417164251742)}}

{'Bagging': {'mean_MAE': np.float64(210665.55989865644),
  'std_MAE': np.float64(3464.5295266417456)},
 'RandomForest': {'mean_MAE': np.float64(202275.86038910906),
  'std_MAE': np.float64(3298.87090728044)},
 'HistGradientBoosting': {'mean_MAE': np.float64(205014.8322832639),
  'std_MAE': np.float64(3348.417164251742)}}

### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




**Answer**

**Adding Engineered Features: Before vs After**

| Model                  | Mean MAE (Before) | Mean MAE (After) | Std MAE (Before) | Std MAE (After) |
|-----------------------|------------------|------------------|------------------|-----------------|
| Bagging               | 209,263          | 210,666          | 3,411            | 3,465           |
| Random Forest         | 201,045          | 202,276          | 3,456            | 3,299           |
| HistGradientBoosting  | 204,429          | 205,015          | 3,478            | 3,348           |


**Did any models show notable improvement in performance?**
- No, none of the models showed improvement after adding the engineered features. In fact, all three models experienced a **slight increase in MAE**, indicating a small decrease in performance. The changes were not very large, but they were consistent across all models, <u>suggesting that the added features did not provide additional useful information for these models</u>.


**Which new features seemed to help — and in which models?**
- Based on the results, none of the new features clearly improved performance for any of the models. Although the interaction feature **sqft_x_bath** was expected to be helpful (since it had a **high F-score in Milestone 1**), it did not lead to better results here. However, there was a **slight reduction in standard deviation** for Random Forest and HistGradientBoosting, which may suggest that the new features contributed to slightly **more stable predictions**, even if accuracy did not improve.


**Do you have any hypotheses about why a particular feature helped (or didn’t)?**
- One likely explanation is that **tree-based models already capture nonlinear relationships and feature interactions automatically**, so manually adding interaction features like **sqft_x_bath** may introduce redundancy rather than new information. Similarly, log transformations are typically more useful for linear models, while tree-based models are less sensitive to skewed distributions. Additionally, adding multiple derived features may increase **multicollinearity** and introduce noise, which can slightly degrade model performance. Overall, this suggests that feature engineering needs to be aligned with the type of model being used, and not all transformations will lead to improvements.

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [13]:
# ---------------------------------------------
# Part 3: Feature Selection
# Using full feature set = original + engineered
# ---------------------------------------------

# Start from the engineered feature dataset
X_train_fs = X_train_fe.copy()

# Scale full feature set
scaler_fs = StandardScaler()
X_train_fs_scaled = pd.DataFrame(
    scaler_fs.fit_transform(X_train_fs),
    columns=X_train_fs.columns,
    index=X_train_fs.index
)

# Same models
models_fs = {
    "Bagging": BaggingRegressor(random_state=random_state),
    "RandomForest": RandomForestRegressor(random_state=random_state),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=random_state)
}

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)

selected_features_by_model = {}
results_selected = {}

for name, model in models_fs.items():
    # Fit model on full feature set
    model.fit(X_train_fs_scaled, y_train)
    
    # Get feature importance
    if hasattr(model, "feature_importances_"):
        importances = pd.Series(model.feature_importances_, index=X_train_fs_scaled.columns)
    elif name == "Bagging":
        # Average feature importances across bagged estimators if available
        estimator_importances = []
        for est in model.estimators_:
            if hasattr(est, "feature_importances_"):
                estimator_importances.append(est.feature_importances_)
        importances = pd.Series(np.mean(estimator_importances, axis=0), index=X_train_fs_scaled.columns)
    else:
        continue
    
    # Keep features with above-average importance
    threshold = importances.mean()
    selected_features = importances[importances > threshold].index.tolist()
    
    selected_features_by_model[name] = selected_features
    
    # Re-run CV using only selected features
    X_selected = X_train_fs_scaled[selected_features]
    
    scores = cross_val_score(
        model,
        X_selected,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1
    )
    
    mae_scores = -scores
    results_selected[name] = {
        "n_features": len(selected_features),
        "selected_features": selected_features,
        "mean_MAE": mae_scores.mean(),
        "std_MAE": mae_scores.std()
    }

pd.DataFrame({
    model: {
        "n_features": results_selected[model]["n_features"],
        "mean_MAE": results_selected[model]["mean_MAE"],
        "std_MAE": results_selected[model]["std_MAE"]
    }
    for model in results_selected
}).T


,n_features,mean_MAE,std_MAE
Bagging,6.0,211196.058384,3724.432487
RandomForest,6.0,203471.169500,3757.182735


In [14]:
for model, feats in selected_features_by_model.items():
    print(f"\n{model} selected features ({len(feats)}):")
    print(feats)


Bagging selected features (6):
['calculatedfinishedsquarefeet', 'latitude', 'longitude', 'yearbuilt', 'log_sqft', 'sqft_x_bath']

RandomForest selected features (6):
['calculatedfinishedsquarefeet', 'latitude', 'longitude', 'yearbuilt', 'log_sqft', 'sqft_x_bath']


In [15]:
from sklearn.feature_selection import SelectKBest, f_regression

# ---------------------------------------------
# HistGradientBoosting feature selection
# ---------------------------------------------
subset_sizes = [5, 6, 8, 10, 12, X_train_fs_scaled.shape[1]]

best_mae = float("inf")
best_std = None
best_features_hgb = None

for k in subset_sizes:
    selector = SelectKBest(score_func=f_regression, k=k)
    X_selected = selector.fit_transform(X_train_fs_scaled, y_train)

    selected_features = X_train_fs_scaled.columns[selector.get_support()].tolist()

    model = HistGradientBoostingRegressor(random_state=random_state)

    scores = cross_val_score(
        model,
        X_selected,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=cv,
        n_jobs=-1
    )

    mae_scores = -scores
    mean_mae = mae_scores.mean()
    std_mae = mae_scores.std()

    if mean_mae < best_mae:
        best_mae = mean_mae
        best_std = std_mae
        best_features_hgb = selected_features

print("Best HistGradientBoosting features:")
print(best_features_hgb)
print("Mean MAE:", best_mae)
print("Std MAE:", best_std)

Best HistGradientBoosting features:
['airconditioningtypeid', 'bathroomcnt', 'bedroomcnt', 'buildingqualitytypeid', 'calculatedfinishedsquarefeet', 'garagecarcnt', 'garagetotalsqft', 'heatingorsystemtypeid', 'latitude', 'longitude', 'lotsizesquarefeet', 'propertylandusetypeid', 'propertyzoningdesc', 'regionidcounty', 'roomcnt', 'unitcnt', 'yearbuilt', 'log_sqft', 'log_lotsize', 'sqft_x_bath', 'bed_bath_ratio']
Mean MAE: 205014.8322832639
Std MAE: 3348.417164251742


In [16]:
part3_results = pd.DataFrame({
    "Bagging": {
        "n_features": 6,
        "mean_MAE": 211196.058384,
        "std_MAE": 3724.432487
    },
    "RandomForest": {
        "n_features": 6,
        "mean_MAE": 203471.169500,
        "std_MAE": 3757.182735
    },
    "HistGradientBoosting": {
        "n_features": len(best_features_hgb),
        "mean_MAE": best_mae,
        "std_MAE": best_std
    }
}).T

part3_results

,n_features,mean_MAE,std_MAE
Bagging,6.0,211196.058384,3724.432487
RandomForest,6.0,203471.169500,3757.182735
HistGradientBoosting,21.0,205014.832283,3348.417164


### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


Answer

Feature Selection Results

Model	Selected Features	Mean MAE	Std MAE
Bagging	6	211,196.06	3,724.43
Random Forest	6	203,471.17	3,757.18
HistGradientBoosting	21	205,014.83	3,348.42

Did performance improve for any models after reducing the number of features?

No, feature selection did not improve performance for any of the models. Both Bagging and Random Forest had slightly higher MAE after reducing the number of features, and HistGradientBoosting performed best when all 21 features were kept. This suggests that reducing the feature set did not help and may have removed information that was still useful for prediction.

Which features were consistently retained across models?

The features consistently retained across Bagging and Random Forest were:

calculatedfinishedsquarefeet
latitude
longitude
yearbuilt
log_sqft
sqft_x_bath

These make sense as important predictors because they capture home size, location, age, and an interaction between square footage and number of bathrooms.

Were any of your newly engineered features selected as important?

Yes, two of the newly engineered features were selected as important by both Bagging and Random Forest:

log_sqft
sqft_x_bath

This suggests that these engineered features did capture some useful information. However, even though they were retained, the reduced feature subsets still did not improve model accuracy overall.> Your text here

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [17]:
# Add as many cells as you need


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


> Your text here

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [18]:
# Add as many cells as you need


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

> Your text here